# Neplish ASR - Whisper Fine-tuning with QLoRA

Fine-tune **OpenAI Whisper Medium** (769M params) on code-switched Nepali-English (Neplish) speech using **QLoRA** (4-bit NF4 quantization + LoRA).

QLoRA lets us fit the full whisper-medium model on a **6 GB GPU** (e.g. RTX 4050) by quantizing the base weights to 4-bit and only training small LoRA adapter layers in fp16.

**Requirements:** GPU with ≥6 GB VRAM. In Colab: Runtime → Change runtime type → GPU (T4 or better).

**Instructions:**
1. Upload your project zip or clone the repo into Colab.
2. Run all cells sequentially.
3. The trained model will be saved to `models/whisper-neplish/final/`.


## 0. Install Dependencies (Colab)


In [1]:
# Uncomment the following lines when running on Google Colab
# !pip install -q transformers datasets peft accelerate evaluate jiwer soundfile librosa bitsandbytes

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch version: 2.10.0+cu128
CUDA available : True
GPU            : NVIDIA GeForce RTX 4050 Laptop GPU
GPU Memory     : 6.1 GB


## 1. Setup and Configuration


In [2]:
import os
import sys
from pathlib import Path
from functools import partial

import evaluate
import numpy as np
import torch
from datasets import load_from_disk, Audio
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    BitsAndBytesConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperFeatureExtractor,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperTokenizer,
)

# Resolve project root
PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /home/gyaneshwar/college/sixth_sem/minor-project


In [3]:
# ---------- Configuration (edit these as needed) ----------

MODEL_NAME = "openai/whisper-medium"  # 769M params, loaded in 4-bit via QLoRA
LANGUAGE = "ne"                       # Nepali
TASK = "transcribe"

# QLoRA: 4-bit quantization settings
LOAD_IN_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"              # NormalFloat4 — best for QLoRA
BNB_4BIT_COMPUTE_DTYPE = torch.float16   # compute dtype for 4-bit layers
BNB_4BIT_USE_DOUBLE_QUANT = True         # nested quantization saves ~0.4 GB

# LoRA hyperparameters (applied on top of quantized base)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# Training hyperparameters (tuned for 6 GB VRAM)
NUM_EPOCHS = 8
BATCH_SIZE = 4            # 4 for 6 GB GPU; increase to 8 on ≥12 GB
GRAD_ACCUM_STEPS = 4      # Effective batch size = 4 × 4 = 16
LEARNING_RATE = 1e-4      # QLoRA benefits from a higher LR
WARMUP_STEPS = 200
EVAL_STEPS = 200
SAVE_STEPS = 200
OPTIM = "paged_adamw_8bit"  # 8-bit optimizer to save memory

# Paths
DATASET_DIR = str(PROJECT_ROOT / "data" / "hf_dataset")
OUTPUT_DIR = str(PROJECT_ROOT / "models" / "whisper-neplish")

print("Configuration loaded.")
print(f"  Model          : {MODEL_NAME}")
print(f"  Quantization   : {'4-bit NF4 (QLoRA)' if LOAD_IN_4BIT else 'None'}")
print(f"  Batch size     : {BATCH_SIZE} × {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"  Optimizer      : {OPTIM}")


Configuration loaded.
  Model          : openai/whisper-medium
  Quantization   : 4-bit NF4 (QLoRA)
  Batch size     : 4 × 4 = 16 effective
  Learning rate  : 0.0001
  Optimizer      : paged_adamw_8bit


## 2. Load Model in 4-bit + Tokenizer and Processor


In [4]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

# --- 4-bit quantization config (QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# Whisper-specific settings
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

print(f"Model loaded: {MODEL_NAME} (4-bit NF4)")
print(f"Total parameters: {model.num_parameters():,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded: openai/whisper-medium (4-bit NF4)
Total parameters: 763,857,920
GPU memory used: 0.60 GB


## 3. Prepare for QLoRA and Apply LoRA Adapters


In [5]:
# Step 1: Prepare the 4-bit model for k-bit training
#   - casts layer norms to fp32 for stability
#   - enables gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Step 2: Apply LoRA adapters on top of the frozen 4-bit weights
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


trainable params: 7,077,888 || all params: 770,935,808 || trainable%: 0.9181
GPU memory after LoRA: 0.63 GB


## 4. Load and Preprocess Dataset


In [14]:
import soundfile as sf
from io import BytesIO
import librosa
import time

dataset = load_from_disk(DATASET_DIR)

# Disable automatic audio decoding to avoid torchcodec dependency
dataset = dataset.cast_column("audio", Audio(decode=False))

print(dataset)

def preprocess(examples):
    """Extract mel features and tokenise the transcript."""
    audio_dict = examples["audio"]

    # Manually decode audio using soundfile
    if audio_dict["bytes"] is not None:
        audio_array, sr = sf.read(BytesIO(audio_dict["bytes"]))
    else:
        audio_array, sr = sf.read(audio_dict["path"])

    # Resample to 16 kHz if needed
    if sr != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
        sr = 16000

    input_features = feature_extractor(
        audio_array,
        sampling_rate=sr,
        return_tensors="np",
    ).input_features[0]
    # Whisper decoder max length is 448 tokens — truncate to avoid ValueError
    labels = tokenizer(
        examples["sentence"], truncation=True, max_length=448
    ).input_ids
    return {"input_features": input_features, "labels": labels}

# ---- Sanity check: preprocess ONE sample directly ----
print("\n--- Sanity check: preprocessing 1 sample ---")
sample = dataset["train"][0]
print(f"  Sentence : {sample['sentence'][:80]}...")
print(f"  Audio keys: {list(sample['audio'].keys())}")
t0 = time.time()
result = preprocess(sample)
elapsed = time.time() - t0
print(f"  input_features shape: {result['input_features'].shape}")
print(f"  labels length       : {len(result['labels'])}")
print(f"  Time for 1 sample   : {elapsed:.2f}s")
print(f"  Estimated total time: ~{elapsed * len(dataset['train']) / 60:.1f} min (train split)")
print("--- Sanity check PASSED! ---\n")

# NOTE: removed num_proc=4 — multiprocessing with audio bytes often causes
# deadlocks (which is why progress was stuck at 0%). Single-process is safer.
print("Preprocessing dataset (this may take a few minutes) ...")
dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names,
)
print("Done!")
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 5440
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 680
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 680
    })
})

--- Sanity check: preprocessing 1 sample ---
  Sentence : यसका लागि, मैले केही संरचित चरणहरू प्रस्ताव गरेको छु । पहिलो चरणमा, हामी सुझाव स...
  Audio keys: ['bytes', 'path']
  input_features shape: (80, 3000)
  labels length       : 300
  Time for 1 sample   : 0.02s
  Estimated total time: ~2.0 min (train split)
--- Sanity check PASSED! ---

Preprocessing dataset (this may take a few minutes) ...


Map:   0%|          | 0/5440 [00:00<?, ? examples/s]

Map:   0%|          | 0/680 [00:00<?, ? examples/s]

Map:   0%|          | 0/680 [00:00<?, ? examples/s]

Done!
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 5440
    })
    validation: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 680
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 680
    })
})


## 5. Data Collator and Metrics


In [11]:
class DataCollatorSpeechSeq2SeqWithPadding:
    """Custom data collator for Whisper fine-tuning."""

    def __init__(self, processor, decoder_start_token_id):
        self.processor = processor
        self.decoder_start_token_id = decoder_start_token_id

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# WER metric
metric_wer = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = metric_wer.compute(predictions=pred_str, references=label_str)
    return {"wer": 100 * wer}

print("Data collator and metrics ready.")


Data collator and metrics ready.


## 6. Training


In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=50,
    logging_first_step=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    push_to_hub=False,
    label_names=["labels"],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Required for QLoRA (frozen base weights)
    optim=OPTIM,
    predict_with_generate=True,
    generation_max_length=225,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Trainer ready. Starting training ...")
trainer.train()


Trainer ready. Starting training ...


Step,Training Loss,Validation Loss


ValueError: Labels' sequence length 456 cannot exceed the maximum allowed length of 448 tokens.

## 7. Save Final Model


In [ ]:
final_dir = os.path.join(OUTPUT_DIR, "final")
os.makedirs(final_dir, exist_ok=True)

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
processor.save_pretrained(final_dir)

print(f"Model saved to {final_dir}")


## 8. Quick Evaluation on Test Set


In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(dataset["test"])
print(f"\nTest Results:")
for key, val in test_results.items():
    print(f"  {key}: {val:.4f}" if isinstance(val, float) else f"  {key}: {val}")
